# 🚀 N4 Rocket Telemetry Analysis

**Purpose:** Post-flight and post-test analysis of rocket telemetry data  
**Author:** Nakuja Project - Recovery Team  
**Last Updated:** January 17, 2026

---

## 📊 What This Notebook Does:

1. **Load Telemetry Data** - From existing logs or uploaded CSV files
2. **Data Validation** - Check for missing values, outliers, and data quality
3. **Flight Profile Analysis** - Altitude, velocity, acceleration over time
4. **Phase Detection** - Identify launch, boost, coast, apogee, descent, landing
5. **Performance Metrics** - Max altitude, max velocity, flight time, etc.
6. **Visualizations** - Interactive plots of all telemetry channels
7. **Export Results** - Summary reports and processed data

---

## 🔧 Setup Instructions:

1. Run the notebook cells in order (top to bottom)
2. Choose data source:
   - **Option A:** Select from existing `telemetry_logs/` folder
   - **Option B:** Upload your own CSV file
3. Review analysis outputs and visualizations
4. Export summary report if needed

## 📦 1. Import Required Libraries

Install missing packages if needed:
```bash
pip install pandas numpy matplotlib seaborn plotly ipywidgets
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import datetime
import warnings
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ All libraries loaded successfully!")
print(f"📊 Pandas version: {pd.__version__}")
print(f"🎨 Plotly version: {px.__version__}")

## 📂 2. Data Source Selection

Choose how to load your telemetry data

In [ ]:
# Define paths
TELEMETRY_DIR = Path("telemetry_logs")
OLD_TELEMETRY_DIR = Path("src/telemetry")

# Find all CSV files in telemetry directories
csv_files = []

if TELEMETRY_DIR.exists():
    csv_files.extend(list(TELEMETRY_DIR.glob("*.csv")))

if OLD_TELEMETRY_DIR.exists():
    csv_files.extend(list(OLD_TELEMETRY_DIR.glob("*.csv")))

# Sort by modification time (newest first)
csv_files = sorted(csv_files, key=lambda x: x.stat().st_mtime, reverse=True)

if csv_files:
    print(f"📁 Found {len(csv_files)} telemetry file(s):")
    for i, f in enumerate(csv_files[:10], 1):  # Show first 10
        size_kb = f.stat().st_size / 1024
        mod_time = datetime.fromtimestamp(f.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
        print(f"  {i}. {f.name} ({size_kb:.1f} KB) - Modified: {mod_time}")
    
    if len(csv_files) > 10:
        print(f"  ... and {len(csv_files) - 10} more")
else:
    print("⚠️ No telemetry CSV files found in telemetry_logs/ or src/telemetry/")
    print("   You can upload your own CSV file in the next cell.")

### Option A: Select Existing File

In [ ]:
# Create dropdown for file selection
if csv_files:
    file_dropdown = widgets.Dropdown(
        options=[(f"{f.name} ({f.stat().st_size / 1024:.1f} KB)", str(f)) for f in csv_files],
        description='Select File:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%')
    )
    
    load_button = widgets.Button(
        description='📊 Load Selected File',
        button_style='success',
        icon='check'
    )
    
    output = widgets.Output()
    
    def on_load_click(b):
        with output:
            output.clear_output()
            selected_file = file_dropdown.value
            print(f"Loading: {selected_file}")
            global df, data_loaded
            df = load_telemetry_data(selected_file)
            data_loaded = True
    
    load_button.on_click(on_load_click)
    
    display(file_dropdown, load_button, output)
    data_loaded = False
else:
    print("No files available. Use Option B to upload a CSV file.")
    data_loaded = False

### Option B: Upload Your Own CSV File

In [ ]:
# File upload widget
uploader = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload CSV:'
)

upload_output = widgets.Output()

def on_upload_change(change):
    with upload_output:
        upload_output.clear_output()
        if uploader.value:
            uploaded_file = list(uploader.value.values())[0]
            content = uploaded_file['content']
            
            # Save temporarily
            temp_path = Path("temp_upload.csv")
            with open(temp_path, 'wb') as f:
                f.write(content)
            
            print(f"✅ Uploaded: {uploaded_file['metadata']['name']}")
            print(f"   Size: {len(content) / 1024:.1f} KB")
            
            global df, data_loaded
            df = load_telemetry_data(str(temp_path))
            data_loaded = True
            
            # Clean up temp file
            temp_path.unlink()

uploader.observe(on_upload_change, names='value')
display(uploader, upload_output)

## 🔍 3. Data Loading & Validation

Load and validate the telemetry data

In [ ]:
def load_telemetry_data(filepath):
    """
    Load telemetry CSV with automatic column detection
    Handles both old format (sparse) and new format (full telemetry)
    """
    try:
        # Try reading with automatic detection
        df = pd.read_csv(filepath)
        
        print(f"\n✅ Successfully loaded: {Path(filepath).name}")
        print(f"   Rows: {len(df):,}")
        print(f"   Columns: {len(df.columns)}")
        print(f"\n📋 Available columns:")
        for col in df.columns:
            non_null = df[col].notna().sum()
            print(f"   • {col}: {non_null:,} non-null values ({non_null/len(df)*100:.1f}%)")
        
        # Convert timestamp columns
        if 'timestamp' in df.columns:
            # Unix timestamp to datetime
            df['datetime'] = pd.to_datetime(df['timestamp'], unit='s', errors='coerce')
        elif 'iso_timestamp' in df.columns:
            # ISO format to datetime
            df['datetime'] = pd.to_datetime(df['iso_timestamp'], errors='coerce')
        elif df.columns[0] not in ['timestamp', 'iso_timestamp']:
            # First column might be timestamp
            try:
                df['datetime'] = pd.to_datetime(df.iloc[:, 0], errors='coerce')
            except:
                df['datetime'] = pd.to_datetime(df.index, errors='coerce')
        
        # Create time elapsed column (seconds from start)
        if 'datetime' in df.columns and df['datetime'].notna().any():
            first_valid = df['datetime'].dropna().iloc[0]
            df['time_elapsed'] = (df['datetime'] - first_valid).dt.total_seconds()
        elif 'timestamp' in df.columns:
            df['time_elapsed'] = df['timestamp'] - df['timestamp'].min()
        else:
            # Use record number or index as time proxy
            if 'record_number' in df.columns:
                df['time_elapsed'] = df['record_number'] * 0.05  # Assume 20Hz = 50ms
            else:
                df['time_elapsed'] = df.index * 0.05
        
        # Convert numeric columns
        numeric_cols = ['ax', 'ay', 'az', 'gx', 'gy', 'gz', 
                       'latitude', 'longitude', 'gps_altitude',
                       'pressure', 'temperature', 'agl_altitude', 'velocity',
                       'kalman_altitude', 'kalman_vertical_velocity',
                       'battery_voltage', 'wifi_rssi']
        
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        return df
        
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return None

# If not loaded via widgets, you can manually load here:
# df = load_telemetry_data("telemetry_logs/telemetry_20250117_143022.csv")
# data_loaded = True

print("\n⏳ Waiting for file selection/upload...")
print("   (Run the cells above to select or upload a file)")

## 📊 4. Data Quality Report

Analyze data completeness and quality

In [ ]:
if data_loaded and df is not None:
    print("="*60)
    print("📊 DATA QUALITY REPORT")
    print("="*60)
    
    # Basic statistics
    print(f"\n📈 Dataset Overview:")
    print(f"   Total Records: {len(df):,}")
    print(f"   Time Span: {df['time_elapsed'].max():.1f} seconds ({df['time_elapsed'].max()/60:.1f} minutes)")
    
    if 'record_number' in df.columns:
        update_rate = len(df) / df['time_elapsed'].max() if df['time_elapsed'].max() > 0 else 0
        print(f"   Average Update Rate: {update_rate:.1f} Hz")
    
    # Missing data analysis
    print(f"\n🔍 Data Completeness:")
    missing_summary = []
    for col in df.columns:
        missing_count = df[col].isna().sum()
        missing_pct = (missing_count / len(df)) * 100
        if missing_pct > 0:
            missing_summary.append((col, missing_count, missing_pct))
    
    if missing_summary:
        missing_summary.sort(key=lambda x: x[2], reverse=True)
        for col, count, pct in missing_summary[:10]:  # Top 10 missing
            bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
            print(f"   {col:30s} {bar} {pct:5.1f}% missing ({count:,} values)")
    else:
        print("   ✅ No missing values detected!")
    
    # Value ranges for key telemetry
    print(f"\n📏 Value Ranges:")
    key_cols = ['agl_altitude', 'kalman_altitude', 'velocity', 'kalman_vertical_velocity', 
                'ax', 'ay', 'az', 'battery_voltage']
    
    for col in key_cols:
        if col in df.columns and df[col].notna().any():
            valid_data = df[col].dropna()
            print(f"   {col:30s} Min: {valid_data.min():8.2f}  Max: {valid_data.max():8.2f}  Mean: {valid_data.mean():8.2f}")
    
    print("\n" + "="*60)
else:
    print("⚠️ No data loaded. Please select or upload a file first.")

## 🚀 5. Flight Phase Detection

Automatically detect launch, boost, coast, apogee, descent, and landing

In [ ]:
def detect_flight_phases(df):
    """
    Detect flight phases based on altitude and velocity
    """
    phases = {}
    
    # Use best available altitude data
    alt_col = None
    if 'kalman_altitude' in df.columns and df['kalman_altitude'].notna().any():
        alt_col = 'kalman_altitude'
    elif 'agl_altitude' in df.columns and df['agl_altitude'].notna().any():
        alt_col = 'agl_altitude'
    elif 'gps_altitude' in df.columns and df['gps_altitude'].notna().any():
        alt_col = 'gps_altitude'
    
    if not alt_col:
        print("⚠️ No altitude data available for phase detection")
        return phases
    
    altitude = df[alt_col].fillna(0)
    
    # Detect launch (altitude > 10m)
    launch_idx = altitude[altitude > 10].index
    if len(launch_idx) > 0:
        phases['launch'] = launch_idx[0]
    
    # Detect apogee (maximum altitude)
    apogee_idx = altitude.idxmax()
    phases['apogee'] = apogee_idx
    phases['max_altitude'] = altitude.max()
    
    # Detect landing (altitude returns to near 0 after apogee)
    if apogee_idx is not None:
        post_apogee = altitude.iloc[apogee_idx:]
        landing_candidates = post_apogee[post_apogee < 20].index
        if len(landing_candidates) > 0:
            phases['landing'] = landing_candidates[0]
    
    # Calculate flight duration
    if 'launch' in phases and 'landing' in phases:
        launch_time = df.loc[phases['launch'], 'time_elapsed']
        landing_time = df.loc[phases['landing'], 'time_elapsed']
        phases['flight_duration'] = landing_time - launch_time
    
    return phases

if data_loaded and df is not None:
    phases = detect_flight_phases(df)
    
    if phases:
        print("="*60)
        print("🚀 FLIGHT PHASE DETECTION")
        print("="*60)
        
        if 'launch' in phases:
            launch_time = df.loc[phases['launch'], 'time_elapsed']
            print(f"\n🛫 Launch Detected:")
            print(f"   Time: T+{launch_time:.1f}s")
            print(f"   Record #: {phases['launch']}")
        
        if 'apogee' in phases:
            apogee_time = df.loc[phases['apogee'], 'time_elapsed']
            print(f"\n🎯 Apogee Reached:")
            print(f"   Altitude: {phases['max_altitude']:.1f}m")
            print(f"   Time: T+{apogee_time:.1f}s")
            print(f"   Record #: {phases['apogee']}")
        
        if 'landing' in phases:
            landing_time = df.loc[phases['landing'], 'time_elapsed']
            print(f"\n🛬 Landing Detected:")
            print(f"   Time: T+{landing_time:.1f}s")
            print(f"   Record #: {phases['landing']}")
        
        if 'flight_duration' in phases:
            print(f"\n⏱️ Total Flight Time: {phases['flight_duration']:.1f}s ({phases['flight_duration']/60:.1f} minutes)")
        
        print("\n" + "="*60)
else:
    print("⚠️ No data loaded.")

## 📈 6. Altitude Profile Visualization

Interactive plot of altitude over time with phase markers

In [ ]:
if data_loaded and df is not None:
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Altitude Profile', 'Vertical Velocity'),
        vertical_spacing=0.12,
        row_heights=[0.6, 0.4]
    )
    
    # Plot altitude
    if 'kalman_altitude' in df.columns:
        fig.add_trace(
            go.Scatter(x=df['time_elapsed'], y=df['kalman_altitude'],
                      mode='lines', name='Kalman Altitude',
                      line=dict(color='blue', width=2)),
            row=1, col=1
        )
    
    if 'agl_altitude' in df.columns:
        fig.add_trace(
            go.Scatter(x=df['time_elapsed'], y=df['agl_altitude'],
                      mode='lines', name='Raw Altitude',
                      line=dict(color='lightblue', width=1)),
            row=1, col=1
        )
    
    # Add phase markers
    if phases:
        if 'launch' in phases:
            t = df.loc[phases['launch'], 'time_elapsed']
            fig.add_vline(x=t, line_dash="dash", line_color="green", 
                         annotation_text="Launch", row=1, col=1)
        
        if 'apogee' in phases:
            t = df.loc[phases['apogee'], 'time_elapsed']
            fig.add_vline(x=t, line_dash="dash", line_color="red", 
                         annotation_text="Apogee", row=1, col=1)
        
        if 'landing' in phases:
            t = df.loc[phases['landing'], 'time_elapsed']
            fig.add_vline(x=t, line_dash="dash", line_color="orange", 
                         annotation_text="Landing", row=1, col=1)
    
    # Plot velocity
    if 'kalman_vertical_velocity' in df.columns:
        fig.add_trace(
            go.Scatter(x=df['time_elapsed'], y=df['kalman_vertical_velocity'],
                      mode='lines', name='Vertical Velocity',
                      line=dict(color='purple', width=2)),
            row=2, col=1
        )
    elif 'velocity' in df.columns:
        fig.add_trace(
            go.Scatter(x=df['time_elapsed'], y=df['velocity'],
                      mode='lines', name='Velocity',
                      line=dict(color='purple', width=2)),
            row=2, col=1
        )
    
    # Update layout
    fig.update_xaxes(title_text="Time (seconds)", row=2, col=1)
    fig.update_yaxes(title_text="Altitude (m)", row=1, col=1)
    fig.update_yaxes(title_text="Velocity (m/s)", row=2, col=1)
    
    fig.update_layout(
        height=800,
        title_text="Flight Profile Analysis",
        showlegend=True,
        hovermode='x unified'
    )
    
    fig.show()
else:
    print("⚠️ No data loaded.")

## 🎯 7. Acceleration Analysis

3-axis acceleration and G-forces

In [ ]:
if data_loaded and df is not None and 'ax' in df.columns:
    fig = go.Figure()
    
    # Plot 3-axis acceleration
    if 'ax' in df.columns:
        fig.add_trace(go.Scatter(x=df['time_elapsed'], y=df['ax'],
                                mode='lines', name='Accel X',
                                line=dict(color='red')))
    
    if 'ay' in df.columns:
        fig.add_trace(go.Scatter(x=df['time_elapsed'], y=df['ay'],
                                mode='lines', name='Accel Y',
                                line=dict(color='green')))
    
    if 'az' in df.columns:
        fig.add_trace(go.Scatter(x=df['time_elapsed'], y=df['az'],
                                mode='lines', name='Accel Z',
                                line=dict(color='blue')))
    
    # Add phase markers
    if phases and 'launch' in phases:
        t = df.loc[phases['launch'], 'time_elapsed']
        fig.add_vline(x=t, line_dash="dash", line_color="green", annotation_text="Launch")
    
    if phases and 'apogee' in phases:
        t = df.loc[phases['apogee'], 'time_elapsed']
        fig.add_vline(x=t, line_dash="dash", line_color="red", annotation_text="Apogee")
    
    fig.update_layout(
        title="3-Axis Acceleration Profile",
        xaxis_title="Time (seconds)",
        yaxis_title="Acceleration (g)",
        height=500,
        hovermode='x unified'
    )
    
    fig.show()
    
    # Statistics
    print("\n📊 Acceleration Statistics:")
    for axis in ['ax', 'ay', 'az']:
        if axis in df.columns:
            data = df[axis].dropna()
            print(f"   {axis.upper()}: Max = {data.max():.2f}g, Min = {data.min():.2f}g, Mean = {data.mean():.2f}g")
else:
    print("⚠️ No acceleration data available.")

## 🔋 8. Battery & System Health

Monitor battery voltage and system status

In [ ]:
if data_loaded and df is not None:
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Battery Voltage', 'WiFi/Signal Strength'),
        vertical_spacing=0.15
    )
    
    # Battery voltage
    if 'battery_voltage' in df.columns:
        fig.add_trace(
            go.Scatter(x=df['time_elapsed'], y=df['battery_voltage'],
                      mode='lines', name='Battery Voltage',
                      line=dict(color='green', width=2)),
            row=1, col=1
        )
        
        # Add critical voltage line (3.3V for LiPo)
        fig.add_hline(y=3.3, line_dash="dot", line_color="red", 
                     annotation_text="Low Battery", row=1, col=1)
    
    # RSSI (signal strength)
    if 'wifi_rssi' in df.columns:
        fig.add_trace(
            go.Scatter(x=df['time_elapsed'], y=df['wifi_rssi'],
                      mode='lines', name='RSSI',
                      line=dict(color='orange', width=2)),
            row=2, col=1
        )
    
    fig.update_xaxes(title_text="Time (seconds)", row=2, col=1)
    fig.update_yaxes(title_text="Voltage (V)", row=1, col=1)
    fig.update_yaxes(title_text="RSSI (dBm)", row=2, col=1)
    
    fig.update_layout(
        height=600,
        title_text="System Health Monitoring",
        showlegend=True,
        hovermode='x unified'
    )
    
    fig.show()
    
    # Battery statistics
    if 'battery_voltage' in df.columns:
        batt = df['battery_voltage'].dropna()
        print("\n🔋 Battery Analysis:")
        print(f"   Start Voltage: {batt.iloc[0]:.2f}V")
        print(f"   End Voltage: {batt.iloc[-1]:.2f}V")
        print(f"   Voltage Drop: {batt.iloc[0] - batt.iloc[-1]:.2f}V")
        print(f"   Minimum: {batt.min():.2f}V")
        if batt.min() < 3.3:
            print("   ⚠️ WARNING: Battery reached critical level!")
else:
    print("⚠️ No data loaded.")

## 🌍 9. GPS Trajectory (if available)

Plot GPS coordinates on a map

In [ ]:
if data_loaded and df is not None and 'latitude' in df.columns and 'longitude' in df.columns:
    # Filter valid GPS coordinates
    gps_data = df[['latitude', 'longitude', 'time_elapsed']].dropna()
    
    if len(gps_data) > 0:
        # Create map
        fig = px.scatter_mapbox(
            gps_data, 
            lat='latitude', 
            lon='longitude',
            color='time_elapsed',
            size_max=15,
            zoom=13,
            mapbox_style="open-street-map",
            title="GPS Flight Path",
            color_continuous_scale="Viridis",
            labels={'time_elapsed': 'Time (s)'}
        )
        
        fig.update_layout(height=600)
        fig.show()
        
        print(f"\n📍 GPS Statistics:")
        print(f"   Valid GPS Points: {len(gps_data)}")
        print(f"   Launch Coordinates: {gps_data['latitude'].iloc[0]:.6f}, {gps_data['longitude'].iloc[0]:.6f}")
        print(f"   Landing Coordinates: {gps_data['latitude'].iloc[-1]:.6f}, {gps_data['longitude'].iloc[-1]:.6f}")
        
        # Calculate horizontal distance (rough estimate)
        from math import radians, cos, sin, asin, sqrt
        
        def haversine(lat1, lon1, lat2, lon2):
            R = 6371000  # Earth radius in meters
            lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
            dlat = lat2 - lat1
            dlon = lon2 - lon1
            a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
            c = 2 * asin(sqrt(a))
            return R * c
        
        distance = haversine(
            gps_data['latitude'].iloc[0], gps_data['longitude'].iloc[0],
            gps_data['latitude'].iloc[-1], gps_data['longitude'].iloc[-1]
        )
        print(f"   Horizontal Distance: {distance:.1f}m")
    else:
        print("⚠️ No valid GPS coordinates found.")
else:
    print("⚠️ No GPS data available.")

## 📝 10. Flight Summary Report

Generate a comprehensive text summary

In [ ]:
if data_loaded and df is not None:
    report = []
    report.append("="*60)
    report.append("🚀 N4 ROCKET FLIGHT SUMMARY REPORT")
    report.append("="*60)
    report.append(f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append(f"\n📊 DATASET INFORMATION:")
    report.append(f"   Total Records: {len(df):,}")
    report.append(f"   Duration: {df['time_elapsed'].max():.1f} seconds")
    
    if phases:
        report.append(f"\n🚀 FLIGHT PHASES:")
        if 'launch' in phases:
            report.append(f"   Launch Time: T+{df.loc[phases['launch'], 'time_elapsed']:.1f}s")
        if 'apogee' in phases:
            report.append(f"   Apogee Time: T+{df.loc[phases['apogee'], 'time_elapsed']:.1f}s")
            report.append(f"   Maximum Altitude: {phases['max_altitude']:.1f}m")
        if 'landing' in phases:
            report.append(f"   Landing Time: T+{df.loc[phases['landing'], 'time_elapsed']:.1f}s")
        if 'flight_duration' in phases:
            report.append(f"   Total Flight Time: {phases['flight_duration']:.1f}s")
    
    # Performance metrics
    report.append(f"\n📈 PERFORMANCE METRICS:")
    
    if 'kalman_vertical_velocity' in df.columns:
        max_vel = df['kalman_vertical_velocity'].max()
        report.append(f"   Maximum Velocity: {max_vel:.1f} m/s ({max_vel * 3.6:.1f} km/h)")
    
    if 'az' in df.columns:
        max_accel = df['az'].max()
        report.append(f"   Maximum Acceleration: {max_accel:.1f}g")
    
    if 'battery_voltage' in df.columns:
        batt = df['battery_voltage'].dropna()
        report.append(f"\n🔋 BATTERY STATUS:")
        report.append(f"   Start: {batt.iloc[0]:.2f}V")
        report.append(f"   End: {batt.iloc[-1]:.2f}V")
        report.append(f"   Minimum: {batt.min():.2f}V")
    
    report.append("\n" + "="*60)
    
    # Print report
    report_text = "\n".join(report)
    print(report_text)
    
    # Save to file
    report_filename = f"flight_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    with open(report_filename, 'w') as f:
        f.write(report_text)
    
    print(f"\n💾 Report saved to: {report_filename}")
else:
    print("⚠️ No data loaded.")

## 💾 11. Export Processed Data

Save cleaned and analyzed data

In [ ]:
if data_loaded and df is not None:
    # Create exports directory
    export_dir = Path("analysis_exports")
    export_dir.mkdir(exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Export full cleaned data
    csv_filename = export_dir / f"processed_telemetry_{timestamp}.csv"
    df.to_csv(csv_filename, index=False)
    print(f"✅ Full data exported: {csv_filename}")
    
    # Export summary statistics
    stats_filename = export_dir / f"statistics_{timestamp}.csv"
    df.describe().to_csv(stats_filename)
    print(f"✅ Statistics exported: {stats_filename}")
    
    # Export phase markers
    if phases:
        phases_filename = export_dir / f"flight_phases_{timestamp}.json"
        import json
        # Convert numpy types to native Python types
        phases_serializable = {k: int(v) if isinstance(v, (np.integer, np.int64)) else float(v) if isinstance(v, (np.floating, np.float64)) else v for k, v in phases.items()}
        with open(phases_filename, 'w') as f:
            json.dump(phases_serializable, f, indent=2)
        print(f"✅ Flight phases exported: {phases_filename}")
    
    print(f"\n📁 All exports saved to: {export_dir.absolute()}")
else:
    print("⚠️ No data loaded.")

## 🎓 12. Custom Analysis (Optional)

Use this cell for your own custom analysis

In [ ]:
# Example: Compare Kalman filtered vs raw altitude
if data_loaded and df is not None:
    print("✏️ Add your custom analysis code here")
    print("\nAvailable data columns:")
    print(df.columns.tolist())
    
    # Your code here
    # Example:
    # fig = px.line(df, x='time_elapsed', y=['kalman_altitude', 'agl_altitude'])
    # fig.show()

---

## 📚 Documentation & Support

**Telemetry Format:** See `research/scripts/server.py` for CSV column definitions  
**Base Station:** Run `python start_basestation_integrated.py` to collect telemetry  
**Issues:** Report bugs on GitHub or contact the Nakuja Recovery Team

**Notebook Features:**
- ✅ Load CSV from file picker or upload
- ✅ Automatic flight phase detection
- ✅ Interactive Plotly visualizations
- ✅ Battery and system health monitoring
- ✅ GPS trajectory mapping
- ✅ Export reports and processed data

**Last Updated:** January 17, 2026